# Phase 10: MVDB / Circle Rate Comparison

This notebook compares AI predictions with MVDB or circle-rate values when the government valuation dataset is available.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.src.config import configure_logging, ensure_directories, settings
from backend.src.mvdb_comparison import (
    build_mvdb_placeholder_summary,
    create_mvdb_comparison_dataset,
    load_mvdb_dataset,
    save_mvdb_data_requirements,
    save_mvdb_outputs,
    summarize_mvdb_vs_ai,
)

configure_logging()
ensure_directories()
PROJECT_ROOT


In [ ]:
import pandas as pd

predictions_df = pd.read_parquet(settings.processed_data_dir / 'valuation_predictions.parquet')
requirements_path = save_mvdb_data_requirements(settings.reports_dir)
predictions_df.shape, requirements_path


In [ ]:
if settings.mvdb_path is None or not settings.mvdb_path.exists():
    summary = build_mvdb_placeholder_summary(settings.mvdb_path)
    comparison_df = None
else:
    mvdb_df = load_mvdb_dataset(settings.mvdb_path)
    comparison_df, mvdb_value_column, join_columns = create_mvdb_comparison_dataset(predictions_df, mvdb_df)
    summary = summarize_mvdb_vs_ai(comparison_df, settings.mvdb_path, join_columns, mvdb_value_column)
summary


In [ ]:
if comparison_df is not None:
    comparison_df.head()
else:
    json.loads((settings.reports_dir / 'mvdb_data_requirements.json').read_text(encoding='utf-8'))


In [ ]:
comparison_path, summary_path = save_mvdb_outputs(summary, settings.reports_dir, comparison_df=comparison_df)
comparison_path, summary_path


In [ ]:
json.loads((settings.reports_dir / 'mvdb_vs_ai_summary.json').read_text(encoding='utf-8'))
